#### **Q4. Viterbi Algorithm for the Nature Primer**
##### 1. Hidden Markov Model (HMM) Configuration:

##### **States:**
- **E**: Exon  
- **5**: 5′ Splice Site  
- **I**: Intron  

##### **Alphabet (Nucleotide Bases):**
`{A, C, G, T}`

##### **State Transition Probabilities:**
> `Start → E` = 1.0  
> `E → E` = 0.9  
> `E → 5` = 0.1  
> `5 → I` = 1.0  
> `I → I` = 0.9  
> `I → End` = 0.1  

##### **Base Emission Probabilities:**

| State | A     | C     | G     | T     |
|--------|--------|--------|--------|--------|
| **E**  | 0.25  | 0.25  | 0.25  | 0.25  |
| **5**  | 0.05  | 0.00  | 0.95  | 0.00  |
| **I**  | 0.40  | 0.10  | 0.10  | 0.40  |

##### 2. Function to calculate the Log Probability of a given Path :


In [1]:
import math

def safe_log(val):
    if val == 0:
        return -math.inf
    else:
        return math.log(val)

def compute_log_likelihood(path, observed_seq):

    if len(path) != len(observed_seq):
        raise ValueError("Path and sequence lengths do not match")

    total_log_prob = 0.0
    previous_state = 'Start'

    for idx in range(len(observed_seq)):
        current_state = path[idx]
        current_symbol = observed_seq[idx]

        transition = trans_probs[previous_state][current_state]
        emission = emit_probs[current_state][current_symbol]

        total_log_prob += safe_log(transition) + safe_log(emission)
        previous_state = current_state

    # Handle final transition to End from I state
    if previous_state == 'I':
        total_log_prob += safe_log(trans_probs[previous_state]['End'])

    return round(total_log_prob, 2)

# Setup states and probability tables
hidden_states = ['E', '5', 'I']
trans_probs = {
    'Start': {'E': 1.0},
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'End': 0.1}
}

emit_probs = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

# Sample path and sequence
hidden_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
dna_seq = "CTTCATGTGAAAGCAGACGTAAGTCA"

# Compute and print result
log_likelihood = compute_log_likelihood(hidden_path, dna_seq)
print(f"Log probability of the given path: {log_likelihood}")


Log probability of the given path: -41.22


##### 3. Viterbi Algorithm Implementation


In [2]:
def run_viterbi(obs_seq):
    seq_len = len(obs_seq)
    dp = [{}]
    trace = {}

    # Initialize first step
    for init_state in ['E']:
        dp[0][init_state] = math.log(trans_probs['Start'][init_state]) + safe_log(emit_probs[init_state][obs_seq[0]])
        trace[init_state] = [init_state]

    # Dynamic programming through the sequence
    for step in range(1, seq_len):
        dp.append({})
        updated_trace = {}

        for curr in hidden_states:
            max_score = -math.inf
            best_prev = None

            for prev in dp[step - 1]:
                if prev in trans_probs and curr in trans_probs[prev]:
                    trans_score = safe_log(trans_probs[prev][curr])
                    emit_score = safe_log(emit_probs[curr][obs_seq[step]])
                    score = dp[step - 1][prev] + trans_score + emit_score

                    if score > max_score:
                        max_score = score
                        best_prev = prev

            if best_prev is not None:
                dp[step][curr] = max_score
                updated_trace[curr] = trace[best_prev] + [curr]

        trace = updated_trace

    # Final step: find the most probable ending state
    highest_score = -math.inf
    best_final_state = None

    for state in hidden_states:
        prob = dp[seq_len - 1].get(state, -math.inf)
        if prob > highest_score:
            highest_score = prob
            best_final_state = state

    return ''.join(trace[best_final_state]), round(highest_score, 2)


# Execute Viterbi on the sample DNA
most_probable_path, viterbi_log_prob = run_viterbi(dna_seq)
print("Most likely path for the given Example Sequence: ", most_probable_path)
print("Viterbi log probability for the most likey path: ", viterbi_log_prob)


Most likely path for the given Example Sequence:  EEEEEEEEEEEEEEEEEEEEEEEEEE
Viterbi log probability for the most likey path:  -38.68


###  Time and Space Complexity  
- Time: O(n·s²) where n = length of sequence , s = number of states  
- Space: O(n·s) to store the Viterbi table and paths  